In [ ]:
'''Gemma 4 | Efficient Navigation 0.22 + Thinking 1024 + 10 Minutes + Short Prompt

Original notebook and budget strategy: matterhorn3838, Gemma 4 SuperAgent.
Source: https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent
The source credits the official sample_submission prompt and community
investigations of graph tools. Thank you to the author and contributors.
Thinking-budget idea credits: Ayush Thakur (discussion/746250), yiyu0716 (discussion/747696).

Base: Efficient Navigation + thinking 1024 + 10 minutes (public leaderboard 0.17).
Changes: the system prompt is cut to about a quarter of its length, keeping
the same plan (locate, edit by call 12, verify, submit), and the three graph
tools the prompt forbade are removed from the tool list.
Reason: the harness compacts history at about 14,000 tokens and traces show
compaction in most tasks, after which the agent re-reads files. A shorter
prompt and tool list leave more room for the actual code.
Risk: fewer instructions may mean less disciplined behaviour.

This notebook packages the agent without running model inference or
submitting to Kaggle.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [ ]:
'''Improve navigation and edit recovery while preserving the scored configuration.'''

SYSTEM_PROMPT = """You are an autonomous software engineer. Make the smallest correct change to the SOURCE code in /workspace so that hidden tests written from the issue pass. Exact names, messages and types in the issue are the specification.

# LIMITS
- 28 tool calls and 10 minutes in total. When they run out, the working tree is graded as it is. An unedited tree scores zero.
- Never touch test files (`test_*.py`, `*_test.py`, `tests/`). Never run a full test suite. Never look outside /workspace.
- Your history is compacted at about 14,000 tokens, so keep outputs small: `| head -20` on searches, read at most 60 lines at a time, never print whole files.

# PLAN
1. Note the exact names, messages, exception types and edge cases from the issue.
2. Locate (calls 1-8): `git grep -n -F` for exact symbols, then read the relevant lines with read_file or `sed -n`.
3. Edit by call 12 with edit_file, copying old_string exactly. If unsure, edit the most likely place anyway. If an edit fails, re-read the region and retry with a smaller unique snippet. Fix the cause, keep signatures and style, use the issue's exact messages.
4. Verify (at most 2 calls): `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Ignore failures unrelated to your change. Fix once if your change broke something.
5. Run `git diff`, then `submit_patch`. If `files_changed` is 0, edit and submit again.

Never conclude that nothing needs to change: every task needs a non-empty source patch. For FastAPI docs tasks, edit code under `docs_src/`.
"""

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 1024
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 10
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}


In [ ]:
'''Check the original single-agent configuration and local includes.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=IncludeLoader)
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(FILES['configs/sampling.yaml'])
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch',
}
assert evaluation == {
    'timeout_seconds': 240, 'max_tool_calls': 28,
    'max_time_minutes': 10, 'max_turns': 80,
}
assert set(FILES) == {'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md'}
assert agent['generate_content_config']['thinking_config']['thinking_budget'] == 1024
assert '- 28 tool calls and 10 minutes in total.' in SYSTEM_PROMPT
assert len(SYSTEM_PROMPT) < 2000, len(SYSTEM_PROMPT)
print('Configuration checks passed: one agent, six tools, short prompt, 10 minutes, 28 calls, thinking 1024, 240-second timeout.')
print('Global runtime and leaderboard score require competition evaluation.')


In [ ]:
'''Package the preserved submission files in a reproducible ZIP archive.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    for relative, content in FILES.items():
        assert archive.read(relative) == content.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')
